# Phase 3 — Prefill, Stop Sequences, and Clean Structured Output

Ask Claude for JSON and you'll usually get:

    Sure! Here's the JSON you requested:
    ```json
    { ... }
    ```
    Let me know if you need anything else!

Useless for `json.loads()`. This phase fixes that with two techniques:

## What you'll learn
1. **Assistant prefill** — start Claude's reply for it, so it continues your text instead of opening with chit-chat
2. **Stop sequences** — force generation to halt when a specific string appears
3. **Combining both** — the standard pattern for getting raw, parseable JSON every time
4. **Bonus** — the tool-based alternative (more reliable, more setup)

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import json
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

## Step 1 — The problem: Claude likes to chat

Watch what happens when we ask for raw JSON without any guardrails.

In [ ]:
response = client.messages.create(
    model=model,
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": "Give me a JSON object with fields name, role, years_experience for a senior backend engineer."
    }],
)

raw = response.content[0].text
print(raw)

# Try to parse it directly — usually fails
try:
    parsed = json.loads(raw)
    print("\nParsed OK!")
except json.JSONDecodeError as e:
    print("\nJSONDecodeError:", e)

Typically you get markdown fences, a sentence before, a sentence after. `json.loads` chokes.

You *could* regex-extract between the ` ```json ` fences, but that's brittle. The clean fix is below.

## Step 2 — Assistant prefill

Trick: add a fake assistant message at the **end** of your `messages` list. Claude sees "I already started my response, let me continue from here" and picks up right where the prefill leaves off.

Watch what happens when we prefill with just an opening brace `{`.

In [ ]:
response = client.messages.create(
    model=model,
    max_tokens=500,
    messages=[
        {"role": "user", "content": "Give me a JSON object with fields name, role, years_experience for a senior backend engineer."},
        {"role": "assistant", "content": "{"}
    ],
)

print("What Claude generated:")
print(response.content[0].text)
print()
print("Stop reason:", response.stop_reason)

Two things to notice:

1. Claude's reply now starts mid-JSON (`"name": ...`) because it thinks it already wrote the `{`.
2. **The `{` is NOT in `response.content[0].text`.** It's in your prefill. To reconstruct the full JSON you must stitch them: `prefill + generated`.

In [ ]:
prefill = "{"
full_json_text = prefill + response.content[0].text
print("Stitched:")
print(full_json_text)
print()

# Still may have trailing text after the closing brace. We fix that next with stop sequences.
try:
    parsed = json.loads(full_json_text)
    print("Parsed:", parsed)
except json.JSONDecodeError as e:
    print("Still failed:", e)

## Step 3 — Stop sequences

You can give the API a list of strings. The moment Claude generates one, generation halts. The matched string itself is **not** included in the output.

Quick demo first, before we use it for JSON:

In [ ]:
response = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[{"role": "user", "content": "Count from one to ten, separated by commas."}],
    stop_sequences=[", five"],
)

print("Output:    ", repr(response.content[0].text))
print("Stop reason:", response.stop_reason)
print("Stopped on: ", response.stop_sequence)

Output should be `'one, two, three, four'` — clean, no trailing artifact, `stop_reason='stop_sequence'`.

## Step 4 — The standard pattern: prefill + stop = clean JSON

Combine the two: prefill with ` ```json ` to force a code fence open, stop on closing ` ``` ` to chop the trailing fluff. Whatever Claude generated in between is your raw JSON.

In [ ]:
def extract_json(user_prompt: str) -> dict:
    """Ask Claude for JSON and parse it. No string-munging required."""
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": "```json\n"}
        ],
        stop_sequences=["```"],
    )
    raw = response.content[0].text
    return json.loads(raw)

In [ ]:
data = extract_json(
    "Extract structured data: 'Maria Lopez, 34, lives in Madrid, works as a data engineer at Acme Corp since 2019.' "
    "Return JSON with fields: full_name, age (int), city, job_title, employer, year_started (int)."
)

print("Parsed object:")
for k, v in data.items():
    print(f"  {k}: {v!r}")

Notice: zero string parsing, zero regex, zero markdown stripping. Just `json.loads`. This is the pattern you'll use any time you need structured output without setting up a full tool schema.

Same trick works for any structured format — swap the fence:

| Goal | Prefill | Stop sequence |
|------|---------|---------------|
| JSON | ` ```json\n ` | ` ``` ` |
| Python code | ` ```python\n ` | ` ``` ` |
| CSV | ` ```csv\n ` | ` ``` ` |
| Plain XML | `<result>` | `</result>` |

## Step 5 — Try it: list of objects

Same function, more complex output. Claude can return arrays, nested objects, whatever — as long as it parses as JSON, you're fine.

In [ ]:
movies = extract_json(
    "Give me a JSON array of 5 sci-fi movies. Each object should have: title, year (int), director, "
    "and a `themes` array of 2-3 short tag strings. Just the array, no wrapper object."
)

print(f"Got {len(movies)} movies:")
for m in movies:
    print(f"  - {m['title']} ({m['year']}) — themes: {', '.join(m['themes'])}")

## Step 6 — Bonus: the tool-based alternative (preview of Phase 5)

Prefill + stop sequence is great for quick extractions. For high-reliability production extractions, there's a more robust path: define a **tool** whose `input_schema` *is* the shape of data you want, then force Claude to call it.

You don't need to understand all of this yet — just see it once so you know it exists. We'll go deep on tool use in Phase 5.

In [ ]:
person_schema = {
    "name": "record_person",
    "description": "Records structured biographical info about one person.",
    "input_schema": {
        "type": "object",
        "properties": {
            "full_name": {"type": "string"},
            "age": {"type": "integer"},
            "city": {"type": "string"},
            "job_title": {"type": "string"},
            "employer": {"type": "string"}
        },
        "required": ["full_name", "age", "city", "job_title", "employer"]
    }
}

response = client.messages.create(
    model=model,
    max_tokens=500,
    tools=[person_schema],
    tool_choice={"type": "tool", "name": "record_person"},
    messages=[{
        "role": "user",
        "content": "Maria Lopez, 34, lives in Madrid, works as a data engineer at Acme Corp."
    }],
)

tool_use_block = next(b for b in response.content if b.type == "tool_use")
print(tool_use_block.input)

What's different here:

- The output is **validated** against `input_schema` by the API. If Claude tries to put a string in `age`, it gets corrected before the response reaches you.
- `tool_choice={"type": "tool", "name": ...}` *forces* the tool call, so Claude can't "just answer in text instead."
- The data comes pre-parsed as `.input` (a dict). No `json.loads` needed.

Trade-off: more setup. For one-off scripts, prefill+stop wins. For a production endpoint that must never crash on bad JSON, the tool path wins.

## Mini-exercise

1. **Code extraction.** Adapt `extract_json` into an `extract_python` function that prefills with ` ```python\n ` and stops on ` ``` `. Use it to ask Claude for a function that reverses a linked list. Verify the output parses with `ast.parse(code)`.
2. **Force a list, not an object.** Modify the prefill so Claude *must* return an array. Hint: prefill with `[` instead of an opening fence — what stop sequence makes sense?
3. **Stress-test the tool path.** In step 6, change the user message to something *missing* an age ("Maria Lopez lives in Madrid…"). What does Claude put in the `age` field? Does the API reject the call?

Reply with what you found and we'll go to **Phase 4: evals** — building a small pipeline that scores prompt quality with objective numbers so you stop guessing whether your prompt got better.